# implementing cupy
This code shows that we can do sparse-dense dot products on the GPU using cupy and cupyx.
This is a first step in allowing a cupy-based implementation of tensorly

In [2]:
# we reload the packages as we make changes frequently
%load_ext autoreload
%autoreload 2

In [3]:
from tensorly_custom.contrib.sparse.decomposition import non_negative_tucker
import tensorly_custom as tl
import numpy as np
import pickle
import torch
import os
from typing import List, Tuple
from utils import DATA_DIR
from typing import Optional, Union
import cupy as cp
import cupyx.scipy.sparse as cpx_sparse
import sparse
import tensorflow as tf

# making the multi_mode_dot function available for gpu

In [4]:
def _to_np(x):
    # Accept NumPy arrays or torch tensors; return NumPy view/copy
    if hasattr(x, "detach"):  # torch.Tensor
        return x.detach().cpu().numpy()
    return x

def _torch_or_tucker_load(path, map_location="cpu"):
    """Tries to load a torch-saved file, if fails, tries pickle."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except RuntimeError:
        with open(path, "rb") as f:
            return pickle.load(f)
def cupy_to_torch_sparse(
    cu_mat: cpx_sparse.spmatrix,
    orig_shape: Optional[Tuple[int, ...]] = None,
    device: Union[str, torch.device] = "cpu",
    dtype: Optional[torch.dtype] = None,
) -> torch.Tensor:
    """
    Convert a CuPy sparse matrix (any format) back to a torch sparse COO tensor.

    If orig_shape is None:
        - The torch tensor is 2D and has the same shape as cu_mat.
    If orig_shape is provided and len(orig_shape) == 2:
        - The torch tensor is 2D with that shape.
    If orig_shape is provided and len(orig_shape) > 2:
        - We treat cu_mat.row as the flattened N-D index and unflatten it
          back to N-D using np.unravel_index, assuming the representation
          created by `torch_sparse_to_cupy`.

    Args:
        cu_mat: CuPy sparse matrix (COO/CSR/CSC, will be converted to COO).
        orig_shape: original tensor shape (for N-D tensors).
        device: target torch device.
        dtype: target dtype for values (defaults to inferred from data).

    Returns:
        torch.sparse_coo_tensor on the requested device.
    """
    # Ensure COO format
    if not cpx_sparse.isspmatrix_coo(cu_mat):
        cu_mat = cu_mat.tocoo()

    row_cp = cu_mat.row
    col_cp = cu_mat.col
    data_cp = cu_mat.data

    # Bring back to NumPy on host
    row_np = cp.asnumpy(row_cp)
    col_np = cp.asnumpy(col_cp)
    data_np = cp.asnumpy(data_cp)

    if orig_shape is None:
        # Simple 2D case, use cu_mat.shape directly
        shape = cu_mat.shape
        indices_np = np.vstack([row_np, col_np])
    else:
        shape = tuple(orig_shape)
        if len(shape) == 2:
            # 2D round-trip
            indices_np = np.vstack([row_np, col_np])
        else:
            # N-D round-trip: row_np contains flattened indices
            flat = row_np
            coords = np.unravel_index(flat, shape)  # tuple of arrays
            indices_np = np.vstack(coords)          # shape: (ndim, nnz)

    indices_t = torch.from_numpy(indices_np).long()
    values_t = torch.from_numpy(data_np)

    if dtype is not None:
        values_t = values_t.to(dtype)

    # Build sparse tensor
    x = torch.sparse_coo_tensor(indices_t, values_t, size=shape)
    x = x.coalesce()
    x = x.to(device)

    return x
def torch_sparse_to_cupy(
    x: torch.Tensor,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Convert a torch sparse COO tensor to a CuPy COO sparse matrix.

    For 2D tensors, the mapping is straightforward.
    For N-D tensors (N>2), we flatten the N-D indices to a single row index
    using np.ravel_multi_index and store them in a (prod(shape), 1) matrix.

    Returns:
        (cupy_coo_matrix, original_shape)
    """
    if not x.is_sparse:
        raise TypeError("torch_sparse_to_cupy expects a torch sparse tensor (COO).")

    x = x.coalesce()  # ensure indices are unique & sorted
    indices = x.indices()          # shape: (ndim, nnz)
    values = x.values()            # shape: (nnz,)
    shape = tuple(x.shape)

    # Move to CPU and NumPy
    indices_np = indices.cpu().numpy()
    values_np = values.cpu().numpy()

    ndim, nnz = indices_np.shape

    if ndim == 2:
        # Direct 2D mapping
        row = indices_np[0]
        col = indices_np[1]
        row_cp = cp.asarray(row)
        col_cp = cp.asarray(col)
        data_cp = cp.asarray(values_np)
        cu_mat = cpx_sparse.coo_matrix((data_cp, (row_cp, col_cp)), shape=shape)
    else:
        # Flatten N-D indices to a single dimension (row index)
        coords = [indices_np[d] for d in range(ndim)]
        flat = np.ravel_multi_index(coords, shape)  # shape: (nnz,)
        flat_cp = cp.asarray(flat)
        data_cp = cp.asarray(values_np)
        # Store as a (prod(shape), 1) matrix: column index always 0
        zero_cp = cp.zeros_like(flat_cp)
        cu_mat = cpx_sparse.coo_matrix(
            (data_cp, (flat_cp, zero_cp)),
            shape=(int(np.prod(shape)), 1),
        )

    return cu_mat, shape


def _role_index(role: str) -> int:
    if role == "verb":
        return 0
    elif role == "subject":
        return 1
    elif role == "object":
        return 2
    else:
        raise ValueError("role must be one of {'verb','subject','object'}")


class TuckerDecomposition:
    """Encapsulating the tucker decomposition (core and factors) and the vocabulary,
    providing methods for scoring, slicing, visualisation, etc."""
    def __init__(self, core, factors: List[torch.Tensor], vocab: dict):
        self.core = core
        self.factors = factors
        self.vocab = vocab

    # --- Construction and loading ---
    @classmethod
    def load_from_disk(cls,
                       dataset: str="karrewiet_vectors_ids",
                       method: str="counting",
                       sparse: bool=False,
                       dims: int=750,
                       rank: int=100,
                       iterations: int=1000,
                       map_location: str="cpu",
                          ) -> "TuckerDecomposition":

        """Loads a precomputed tucker decomposition from disk.
            Args:
                dataset (str): name of the dataset
                method (str): method used to compute the decomposition
                    - one of "counting", "sc", "sii"
                dims (int): dimensionality of the original tensor modes (vocab size)
                rank (int): rank of the decomposition
                iterations (int): number of iterations used to compute the decomposition
                map_location (str): device to map the loaded tensors to
            Returns:
                ((core, factors), vocab)
                    core: torch.Tensor
                    factors: list[torch.Tensor]
                    vocab: dict with keys 'vocab_v','vocab_s','vocab_o','v2i','s2i','o2i'
        """
        if method not in {"counting", "sc", "sii"}:
            raise ValueError("method must be one of {'counting','sc','sii'}")
        base = os.path.join(DATA_DIR, "tensors", dataset)

        vocab_path = os.path.join(base, f"vocabularies/{dims}.pkl")
        decomp_path = os.path.join(base,f"{'sparse'*sparse}" , "decomposition", f"{method}_{dims}d_{rank}r_{iterations}i.{'pt' if not sparse else 'pkl'}")
        if not os.path.exists(vocab_path):
            raise FileNotFoundError(f"Missing vocab file: {vocab_path}")
        if not os.path.exists(decomp_path):
            raise FileNotFoundError(f"Missing decomposition file: {decomp_path}")
        # the vocab is here under f"vocabularies_[dims].pkl"
        # Load with torch (they were saved with torch.save)
        with open(vocab_path, "rb") as f:
            vocab = pickle.load(f)
        (core, factors) = _torch_or_tucker_load(decomp_path, map_location=map_location)

        return cls(core, factors, vocab)

    def fetch_latents(self, triple: Tuple[str, str, str]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Fetches the latent representations for a given (verb, subject, object) triple."""
        # triple = (verb_str, subj_str, obj_str)
        v_idx = self.vocab["v2i"][triple[0]]
        s_idx = self.vocab["s2i"][triple[1]]
        o_idx = self.vocab["o2i"][triple[2]]
        V, S, O = [ _to_np(F) for F in self.factors]     # shapes (DIMS,R)
        v = V[v_idx]                                     # (R,)
        s = S[s_idx]                                     # (R,)
        o = O[o_idx]                                     # (R,)
        return v, s, o

    def _core_np(self):
        return _to_np(self.core)

    # -- Sparsity methods ---
    def sparse_representation(self, sparse_type):
        # we return the sparse representation of the tensor
        # we check if our tensor is a tensorflow tensor or make it one
        if sparse_type == "tensorflow":
            if not isinstance(self.core, tf.Tensor):
                core = tf.convert_to_tensor(self.core)
            else:
                core = self.core
            sparse_core = tf.sparse.from_dense(core)
            # we do the same for the factors
            sparse_factors = []
            for factor in self.factors:
                if not isinstance(factor, tf.Tensor):
                    factor = tf.convert_to_tensor(factor)
                sparse_factor = tf.sparse.from_dense(factor)
                sparse_factors.append(sparse_factor)

            return sparse_core, sparse_factors
        elif sparse_type == "torch":
            if not isinstance(self.core, torch.Tensor):
                core = torch.tensor(self.core)
            else:
                core = self.core
            sparse_core = core.to_sparse()
            # we do the same for the factors
            sparse_factors = []
            for factor in self.factors:
                if not isinstance(factor, torch.Tensor):
                    factor = torch.tensor(factor)
                sparse_factor = factor.to_sparse()
                sparse_factors.append(sparse_factor)
            return sparse_core, sparse_factors
        elif sparse_type == "sparse":
            # can only work from a sparse torch tensor
            if not isinstance(self.core, torch.Tensor) or not self.core.is_sparse:
                raise TypeError("sparse_representation expects self.core to be a torch sparse tensor.")
            coords = self.core.indices().numpy()       # shape (nnz, ndim)
            data   = self.core.values().numpy()        # shape (nnz,)
            shape  = tuple(self.core.size())  # e.g. (d0, d1, ..., d_{n-1})
            sparse_core = sparse.COO(coords, data, shape=shape)

            # we do the same for the factors
            sparse_factors = []
            for factor in self.factors:
                if not isinstance(factor, torch.Tensor) or not factor.is_sparse:
                    raise TypeError("sparse_representation expects self.factors to be torch sparse tensors.")
                coords = factor.indices().numpy()       # shape (nnz, ndim)
                data   = factor.values().numpy()        # shape (nnz,)
                shape  = tuple(factor.size())  # e.g. (d0, d1, ..., d_{n-1})
                sparse_factor = sparse.COO(coords, data, shape=shape)
                sparse_factors.append(sparse_factor)

            return sparse_core, sparse_factors
        else:
            raise NotImplementedError(f"Sparse representation for type {sparse_type} not implemented.")



    def tensor_to_sparse(self, sparse_type="tensorflow"):
        self.core, self.factors = self.sparse_representation(sparse_type)


    def tensor_to_dense(self):
        self.core = self._core_np().todense()
        self.factors = [_to_np(factor).todense() for factor in self.factors]

    def core_to_cupy(self):
        """
        Convert self.core (torch sparse) to a CuPy COO matrix, keeping track of shape.
        """
        if not isinstance(self.core, torch.Tensor) or not self.core.is_sparse:
            raise TypeError("core_to_cupy expects self.core to be a torch sparse tensor.")
        self._core_cupy, self._core_shape = torch_sparse_to_cupy(self.core)
        return self._core_cupy

    def core_from_cupy(self):
        """
        Convert self._core_cupy back to a torch sparse tensor with the original shape.
        """
        if not hasattr(self, "_core_cupy") or not hasattr(self, "_core_shape"):
            raise RuntimeError("core_to_cupy must be called before core_from_cupy.")
        self.core = cupy_to_torch_sparse(self._core_cupy, self._core_shape, device=self.core.device, dtype=self.core.dtype)
        return self.core





In [5]:

small_tensor = TuckerDecomposition.load_from_disk(
    dataset="fineweb_old",
    method="sii",
    sparse=False,
    dims=500,
    rank=100,
    iterations=1000,
    map_location="cpu"
)
small_tensor.tensor_to_sparse("torch")
sparse_core, sparse_factors = small_tensor.sparse_representation("sparse")

In [6]:
from tensorly_custom.contrib.sparse.backend import tucker_to_tensor
from tensorly_custom.contrib.sparse.tenalg import multi_mode_dot
reconstructed = tucker_to_tensor((sparse_core, sparse_factors))

tucker_to_tensor time: 46.01281952857971


# Plan: implementing this on cuda!

In [7]:
import cupy as cp
import cupyx
import sparse

# Convert PyData/Sparse COO tensor to CuPy sparse matrix
def pydatasparse_to_cupy_sparse(ps: sparse.COO) -> cupyx.scipy.sparse.coo_matrix:
    """Converts a PyData/Sparse COO tensor to a CuPy sparse matrix."""
    coords = cp.array(ps.coords)  # shape (ndim, nnz)
    data   = cp.array(ps.data)    # shape (nnz,)
    shape  = tuple(ps.shape)      # e.g. (d0, d1, ..., d_{n-1})
    # print(coords.shape, data.shape, shape)
    # we make sure that three-dimensional tensors are converted to 1D vectors
    if len(shape) == 3:
        # we reshape the coords and shape accordingly
        coords = cp.vstack([
            coords[0],
            coords[1] * shape[2] + coords[2]
        ])
        shape = (shape[0], shape[1] * shape[2])
    return cupyx.scipy.sparse.coo_matrix((data, coords), shape=shape)
# Convert the sparse core and factors to CuPy sparse matrices
cupy_sparse_core = pydatasparse_to_cupy_sparse(sparse_core)
cupy_sparse_factors = [pydatasparse_to_cupy_sparse(factor) for factor in sparse_factors]


In [8]:
# we print the shapes to verify
print("Core shape:", cupy_sparse_core.shape)
for i, factor in enumerate(cupy_sparse_factors):
    print(f"Factor {i} shape:", factor.shape)
dense_factor = small_tensor.factors[0].to_dense()
# we need this to be a matrix
dense_factor = dense_factor.numpy()

Core shape: (100, 10000)
Factor 0 shape: (500, 100)
Factor 1 shape: (500, 100)
Factor 2 shape: (500, 100)


In [9]:
cupy_sparse_core    = pydatasparse_to_cupy_sparse(sparse_core)
cupy_sparse_factors = [pydatasparse_to_cupy_sparse(factor) for factor in sparse_factors]

factor0_sparse = cupy_sparse_factors[0]      # shape (500, 100)
A = factor0_sparse.toarray()                 # make it dense on GPU

Y_mode1 = A @ cupy_sparse_core               # (500 x 10000)
print(Y_mode1.shape)
Y = Y_mode1.reshape(500, 100, 100)

(500, 10000)


In [14]:
import cupy as cp
import cupyx
from cupyx.scipy.sparse import coo_matrix
import sparse  # PyData/Sparse


# Convert PyData/Sparse COO tensor to CuPy sparse matrix
def pydatasparse_to_cupy_sparse(ps: sparse.COO) -> coo_matrix:
    """Converts a PyData/Sparse COO tensor to a CuPy sparse matrix.

    For 3D inputs of shape (I, J, K), this returns a 2D matrix of shape
    (I, J*K), corresponding to the mode-1 unfolding.
    """
    coords = cp.array(ps.coords)  # shape (ndim, nnz)
    data   = cp.array(ps.data)    # shape (nnz,)
    shape  = tuple(ps.shape)      # e.g. (d0, d1, ..., d_{n-1})

    # we make sure that three-dimensional tensors are converted to 2D matrices
    # (mode-1 unfolding): (I, J, K) -> (I, J*K)
    if len(shape) == 3:
        I, J, K = shape
        coords = cp.vstack([
            coords[0],               # i
            coords[1] * K + coords[2]  # j*K + k
        ])
        shape = (I, J * K)           # 100 x (100*100) = 100 x 10000

    return coo_matrix((data, coords), shape=shape)


# --- Example: core tensor (100 x 100 x 100) and factor (500 x 100) --- #

# sparse_core: PyData/Sparse COO with shape (100, 100, 100)
# factor:      PyData/Sparse COO or numpy/cupy array with shape (500, 100)

# 1. Convert the sparse core to a CuPy sparse matrix (mode-1 unfolding)
cupy_sparse_core = pydatasparse_to_cupy_sparse(sparse_core)
# cupy_sparse_core.shape == (100, 100 * 100) == (100, 10000)

I, J, K = sparse_core.shape  # (100, 100, 100)
print(I,J,K)
print('tensor shape', cupy_sparse_core.shape)

# 2. Convert the factor to a CuPy *dense* matrix of shape (500, 100)
#    If your factor is PyData/Sparse COO:
# factor_ps: sparse.COO with shape (500, 100)
factor_ps = sparse_factors[0]  # or however you pick your factor
print("factor ps shape", factor_ps.shape)
R, I_factor = factor_ps.shape
print(R, I_factor)
print("I", I)
assert I_factor == I  # sanity check: inner dim must match core's first mode

# get a dense CuPy array for the factor
# (factor_ps.todense() -> numpy array; cp.asarray moves it to GPU)
A = cp.asarray(factor_ps.todense())  # shape (500, 100)
print("A shape:", A.shape)
# 3. Perform the sparse–dense matmul:
#    (500 x 100) @ (100 x 10000) -> (500 x 10000)
Y_mode1 = A @ cupy_sparse_core
print("Y_mode1 shape:", Y_mode1.shape)  # should be (500, 10000)
# Result is a sparse matrix type; convert to dense since the product is generally dense
# 4. Refold back to a 3D tensor: (500, 100, 100)
Y = Y_mode1.reshape(R, J, K)  # shape (500, 100, 100)
print("Y shape:", Y.shape)

100 100 100
tensor shape (100, 10000)
factor ps shape (500, 100)
500 100
I 100
A shape: (500, 100)
Y_mode1 shape: (500, 10000)
Y shape: (500, 100, 100)


In [13]:
# we check if this is the same as doing it with full dense tensors
from tensorly import tucker_to_tensor
import tensorly
tensorly.set_backend("pytorch")
small_tensor_dense = TuckerDecomposition.load_from_disk(
    dataset="fineweb_old",
    method="sii",
    sparse=False,
    dims=500,
    rank=100,
    iterations=1000,
    map_location="cpu"
)

In [14]:
# we perform the same operations on the dense counterparts
core_dense = small_tensor_dense.core      # shape (100, 100, 100)
factor0_dense = small_tensor_dense.factors[0]  # shape (500, 100)
print(core_dense.shape, factor0_dense.shape)
# we do the dot product
Y_mode1_dense_full = torch.matmul(factor0_dense, core_dense.reshape(100, 10000))
Y_mode1_dense_full = Y_mode1_dense_full.reshape(500, 100, 100)

torch.Size([100, 100, 100]) torch.Size([500, 100])


In [15]:
# we check if they are similar
# we convert both to the same format first
y_torch = torch.tensor(Y.get())  # cupy to torch
print(y_torch.shape, Y_mode1_dense_full.shape)
# we compute the difference
difference = np.abs(y_torch - Y_mode1_dense_full)

torch.Size([500, 100, 100]) torch.Size([500, 100, 100])


In [16]:
# we print the largest differences
print("Max difference:", difference.max().item())

Max difference: 6.984919309616089e-10


In [17]:
# for comparison, the max value in each of the tensors
print("Max value in sparse-based Y:", y_torch.max().item())
print("Max value in dense-based Y:", Y_mode1_dense_full.max().item())

Max value in sparse-based Y: 0.05885610729455948
Max value in dense-based Y: 0.05885610729455948


In [18]:
from tensorly.tenalg import multi_mode_dot as mmd
core_dense = small_tensor_dense.core          # torch, (100, 100, 100)
factor0_dense = small_tensor_dense.factors[0] # torch, (500, 100)

reconstructed_dense_disk = mmd(core_dense, [factor0_dense], modes=[0])

Y_np = cp.asnumpy(Y)
y_torch = torch.from_numpy(Y_np).to(reconstructed_dense_disk.dtype)

diff_disk = (y_torch - reconstructed_dense_disk).abs()
print("max abs diff vs disk:", diff_disk.max().item())


max abs diff vs disk: 6.984919309616089e-10


## Native Cupy support
There is native cupy support in Tensorly that we can build upon.
It only works on dense tensors for now, but we could extend it to sparse tensors.

In [19]:
import tensorly as tl

tl.set_backend("cupy")
core_cupy = tl.tensor(sparse_core.todense())  # cupy, (100, 100, 100)
factor0_cupy = tl.tensor(sparse_factors[0].todense())  #
# we perform the multi_mode_dot on cupy tensors
reconstructed_cupy = tl.tenalg.multi_mode_dot(core_cupy, [factor0_cupy], modes=[0])
# we check the difference with the previous result
Y_np = cp.asnumpy(Y)
y_cupy = tl.tensor(Y_np)
diff_cupy = (y_cupy - reconstructed_cupy)
diff_cupy

array([[[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  4.9303807e-32,  0.0000000e+00],
        ...,
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
         -0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00]],

       [[-8.2718061e-25,  0.0000000e+00, -3.3087225e-24, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00, -1.0339758e-25, ...,
          0.0000000e+00,  4.1359031e-25,  0.0000000e+00],
        [ 6.6174449e-24, 

In [20]:
import tensorly_custom.backend as T
# from tensorly_custom import unfold, fold, vec_to_tensor
# --------- basic tensor ops: unfold / fold / vec_to_tensor ----------

def unfold(tensor: cp.ndarray, mode: int) -> cp.ndarray:
    """Mode-n unfolding of a CuPy tensor.

    tensor: shape (i_0, ..., i_{N-1})
    returns: shape (i_mode, prod_{k != mode} i_k)
    """
    tensor = cp.moveaxis(tensor, mode, 0)         # mode -> axis 0
    return tensor.reshape(tensor.shape[0], -1)    # (i_mode, -1)


def fold(unfolded: cp.ndarray, mode: int, shape) -> cp.ndarray:
    """Inverse of unfold_cupy.

    unfolded: shape (i_mode, prod_{k != mode} i_k)
    shape: original full shape (i_0, ..., i_{N-1})
    """
    shape = tuple(shape)
    i_mode = shape[mode]

    # shape with mode dim first, others in order
    other_dims = [shape[i] for i in range(len(shape)) if i != mode]
    tensor = unfolded.reshape((i_mode, *other_dims))
    return cp.moveaxis(tensor, 0, mode)


def vec_to_tensor(vec: cp.ndarray, shape) -> cp.ndarray:
    """Reshape vector to tensor with given shape."""
    return vec.reshape(shape)

def mode_dot(tensor, matrix_or_vector, mode, transpose=False):
    """n-mode product of a tensor and a matrix or vector at the specified mode

    Mathematically: :math:`\\text{tensor} \\times_{\\text{mode}} \\text{matrix or vector}`


    Parameters
    ----------
    tensor : ndarray
        tensor of shape ``(i_1, ..., i_k, ..., i_N)``
    matrix_or_vector : ndarray
        1D or 2D array of shape ``(J, i_k)`` or ``(i_k, )``
        matrix or vectors to which to n-mode multiply the tensor
    mode : int
    transpose : bool, default is False
        If True, the matrix is transposed.
        For complex tensors, the conjugate transpose is used.

    Returns
    -------
    ndarray
        `mode`-mode product of `tensor` by `matrix_or_vector`
        * of shape :math:`(i_1, ..., i_{k-1}, J, i_{k+1}, ..., i_N)` if matrix_or_vector is a matrix
        * of shape :math:`(i_1, ..., i_{k-1}, i_{k+1}, ..., i_N)` if matrix_or_vector is a vector

    See also
    --------
    multi_mode_dot : chaining several mode_dot in one call
    """
    # the mode along which to fold might decrease if we take product with a vector
    fold_mode = mode
    new_shape = list(tensor.shape)

    if T.ndim(matrix_or_vector) == 2:  # Tensor times matrix
        # Test for the validity of the operation
        dim = 0 if transpose else 1
        if matrix_or_vector.shape[dim] != tensor.shape[mode]:
            raise ValueError(
                f"shapes {tensor.shape} and {matrix_or_vector.shape} not aligned in mode-{mode} multiplication: "
                f"{tensor.shape[mode]} (mode {mode}) != {matrix_or_vector.shape[dim]} (dim 1 of matrix)"
            )

        if transpose:
            matrix_or_vector = T.conj(T.transpose(matrix_or_vector))

        new_shape[mode] = matrix_or_vector.shape[0]
        vec = False

    elif T.ndim(matrix_or_vector) == 1:  # Tensor times vector
        if matrix_or_vector.shape[0] != tensor.shape[mode]:
            raise ValueError(
                f"shapes {tensor.shape} and {matrix_or_vector.shape} not aligned for mode-{mode} multiplication: "
                f"{tensor.shape[mode]} (mode {mode}) != {matrix_or_vector.shape[0]} (vector size)"
            )
        if len(new_shape) > 1:
            new_shape.pop(mode)
        else:
            new_shape = ()
        vec = True

    else:
        raise ValueError(
            "Can only take n_mode_product with a vector or a matrix."
            f"Provided array of dimension {T.ndim(matrix_or_vector)} not in [1, 2]."
        )

    res = T.dot(matrix_or_vector, unfold(tensor, mode))

    if vec:  # We contracted with a vector, leading to a vector
        return vec_to_tensor(res, shape=new_shape)
    else:  # tensor times vec: refold the unfolding
        return fold(res, fold_mode, new_shape)


def multi_mode_dot(tensor, matrix_or_vec_list, modes=None, skip=None, transpose=False):
    """n-mode product of a tensor and several matrices or vectors over several modes

    Parameters
    ----------
    tensor : ndarray

    matrix_or_vec_list : list of matrices or vectors of length ``tensor.ndim``

    skip : None or int, optional, default is None
        If not None, index of a matrix to skip.
        Note that in any case, `modes`, if provided, should have a length of ``tensor.ndim``

    modes : None or int list, optional, default is None

    transpose : bool, optional, default is False
        If True, the matrices or vectors in in the list are transposed.
        For complex tensors, the conjugate transpose is used.

    Returns
    -------
    ndarray
        tensor times each matrix or vector in the list at mode `mode`

    Notes
    -----
    If no modes are specified, just assumes there is one matrix or vector per mode and returns:

    :math:`\\text{tensor  }\\times_0 \\text{ matrix or vec list[0] }\\times_1 \\cdots \\times_n \\text{ matrix or vec list[n] }`

    See also
    --------
    mode_dot
    """
    if modes is None:
        modes = range(len(matrix_or_vec_list))

    decrement = 0  # If we multiply by a vector, we diminish the dimension of the tensor

    res = tensor

    # Order of mode dots doesn't matter for different modes
    # Sorting by mode shouldn't change order for equal modes
    factors_modes = sorted(zip(matrix_or_vec_list, modes), key=lambda x: x[1])
    for i, (matrix_or_vec, mode) in enumerate(factors_modes):
        if (skip is not None) and (i == skip):
            continue

        if transpose:
            res = mode_dot(res, T.conj(T.transpose(matrix_or_vec)), mode - decrement)
        else:
            res = mode_dot(res, matrix_or_vec, mode - decrement)

        if T.ndim(matrix_or_vec) == 1:
            decrement += 1

    return res

In [21]:
# we check if this works out of the box on the sparse cupy tensors
reconstructed_cupy_custom = multi_mode_dot(cupy_sparse_core, [cupy_sparse_factors[0]], modes=[0])
# we check the difference with the previous result

TypeError: Argument 'a' has incorrect type (expected cupy._core.core._ndarray_base, got coo_matrix)

In [1]:
import cupy as cp
import cupyx
from cupyx.scipy.sparse import spmatrix as CupySparseMatrix

import tensorly_custom.backend as T


def _is_cupy_sparse(x):
    return isinstance(x, CupySparseMatrix)


# --------- basic tensor ops: unfold / fold / vec_to_tensor ----------

def unfold(tensor, mode: int):
    """Mode-n unfolding.

    For dense CuPy arrays:
        (i_0, ..., i_N-1) -> (i_mode, prod_{k != mode} i_k)

    For CuPy sparse matrices (2D):
        - mode == 0: return as-is (already unfolded along rows)
        - mode == 1: return transpose
    """
    if _is_cupy_sparse(tensor):
        # We only support 2D sparse for now
        if mode == 0:
            return tensor
        elif mode == 1:
            return tensor.T
        else:
            raise ValueError(
                f"unfold for sparse tensors only supports mode 0 or 1. Got mode={mode}"
            )

    # dense path (same as you had)
    tensor = cp.moveaxis(tensor, mode, 0)         # mode -> axis 0
    return tensor.reshape(tensor.shape[0], -1)    # (i_mode, -1)


def fold(unfolded, mode: int, shape):
    """Inverse of unfold.

    For dense:
        undo `unfold`.

    For sparse:
        we assume 2D and just return as-is (no meaningful folding on GPU sparse).
    """
    if _is_cupy_sparse(unfolded):
        # In your sparse path, you're working with a 2D unfolded core anyway.
        # Folding it back to 3D is done later with `.reshape` on a dense array.
        return unfolded

    shape = tuple(shape)
    i_mode = shape[mode]

    # shape with mode dim first, others in order
    other_dims = [shape[i] for i in range(len(shape)) if i != mode]
    tensor = unfolded.reshape((i_mode, *other_dims))
    return cp.moveaxis(tensor, 0, mode)


def vec_to_tensor(vec, shape):
    """Reshape vector to tensor with given shape."""
    return vec.reshape(shape)


# ---------------------- mode_dot with sparse support ----------------------

def mode_dot(tensor, matrix_or_vector, mode, transpose=False):
    """n-mode product of a tensor and a matrix or vector at the specified mode.

    Matches TensorLy semantics, but with extra support for:
        - tensor as a 2D CuPy sparse matrix (already unfolded)
        - matrix_or_vector as dense or sparse
    """
    fold_mode = mode
    new_shape = list(tensor.shape)

    # --- shape / type checks via backend T for dense, but fine for sparse too ---

    if T.ndim(matrix_or_vector) == 2:  # Tensor times matrix
        dim = 0 if transpose else 1
        if matrix_or_vector.shape[dim] != tensor.shape[mode]:
            raise ValueError(
                f"shapes {tensor.shape} and {matrix_or_vector.shape} not aligned in mode-{mode} multiplication: "
                f"{tensor.shape[mode]} (mode {mode}) != {matrix_or_vector.shape[dim]} (dim {dim} of matrix)"
            )

        if transpose:
            matrix_or_vector = T.conj(T.transpose(matrix_or_vector))

        new_shape[mode] = matrix_or_vector.shape[0]
        vec = False

    elif T.ndim(matrix_or_vector) == 1:  # Tensor times vector
        if matrix_or_vector.shape[0] != tensor.shape[mode]:
            raise ValueError(
                f"shapes {tensor.shape} and {matrix_or_vector.shape} not aligned for mode-{mode} multiplication: "
                f"{tensor.shape[mode]} (mode {mode}) != {matrix_or_vector.shape[0]} (vector size)"
            )
        if len(new_shape) > 1:
            new_shape.pop(mode)
        else:
            new_shape = ()
        vec = True

    else:
        raise ValueError(
            "Can only take n_mode_product with a vector or a matrix."
            f"Provided array of dimension {T.ndim(matrix_or_vector)} not in [1, 2]."
        )

    # --- unfolding ---
    unfolded = unfold(tensor, mode)

    # --- dot product ---
    # If any operand is sparse, use @ so CuPy's sparse machinery handles it.
    if _is_cupy_sparse(unfolded) or _is_cupy_sparse(matrix_or_vector):
        res = matrix_or_vector @ unfolded
    else:
        res = T.dot(matrix_or_vector, unfolded)

    # --- vector vs matrix case ---
    if vec:  # contracted with a vector → result is a vector
        return vec_to_tensor(res, shape=new_shape)
    else:    # tensor times matrix: refold
        return fold(res, fold_mode, new_shape)


# ---------------- multi_mode_dot (works with sparse core, mode 0) -----------

def multi_mode_dot(tensor, matrix_or_vec_list, modes=None, skip=None, transpose=False):
    """n-mode product of a tensor and several matrices or vectors over several modes

    This now works when:
      - `tensor` is a 2D CuPy sparse matrix (your unfolded core),
      - you multiply along mode 0 (like in your example),
      - matrices/vectors can be dense or sparse.
    """
    if modes is None:
        modes = range(len(matrix_or_vec_list))

    decrement = 0  # If we multiply by a vector, we diminish the dimension of the tensor

    res = tensor

    # Order of mode dots doesn't matter for different modes
    factors_modes = sorted(zip(matrix_or_vec_list, modes), key=lambda x: x[1])
    for i, (matrix_or_vec, mode) in enumerate(factors_modes):
        if (skip is not None) and (i == skip):
            continue

        if transpose:
            res = mode_dot(res, T.conj(T.transpose(matrix_or_vec)), mode - decrement)
        else:
            res = mode_dot(res, matrix_or_vec, mode - decrement)

        if T.ndim(matrix_or_vec) == 1:
            decrement += 1

    return res


cupy_sparse_core = pydatasparse_to_cupy_sparse(sparse_core)       # (100, 10000)
cupy_sparse_factors = [pydatasparse_to_cupy_sparse(f) for f in sparse_factors]
# cupy_sparse_factors[0]: (500, 100)
I, J, K = sparse_core.shape  # (100, 100, 100)
R = cupy_sparse_factors[0].shape[0]  # 500

NameError: name 'pydatasparse_to_cupy_sparse' is not defined

In [23]:
# mode-0 product on the unfolded sparse core
reconstructed_cupy_custom = multi_mode_dot(
    cupy_sparse_core, [cupy_sparse_factors[0]], modes=[0]
)
print(reconstructed_cupy_custom.shape)  # (500, 10000)  -> still unfolded

(500, 10000)


In [ ]:


def mode_dot(tensor, matrix_or_vector, mode, transpose=False):
    """n-mode product of a tensor and a matrix or vector at the specified mode

    Mathematically: :math:`\\text{tensor} \\times_{\\text{mode}} \\text{matrix or vector}`


    Parameters
    ----------
    tensor : ndarray
        tensor of shape ``(i_1, ..., i_k, ..., i_N)``
    matrix_or_vector : ndarray
        1D or 2D array of shape ``(J, i_k)`` or ``(i_k, )``
        matrix or vectors to which to n-mode multiply the tensor
    mode : int
    transpose : bool, default is False
        If True, the matrix is transposed.
        For complex tensors, the conjugate transpose is used.

    Returns
    -------
    ndarray
        `mode`-mode product of `tensor` by `matrix_or_vector`
        * of shape :math:`(i_1, ..., i_{k-1}, J, i_{k+1}, ..., i_N)` if matrix_or_vector is a matrix
        * of shape :math:`(i_1, ..., i_{k-1}, i_{k+1}, ..., i_N)` if matrix_or_vector is a vector

    See also
    --------
    multi_mode_dot : chaining several mode_dot in one call
    """
    # the mode along which to fold might decrease if we take product with a vector
    fold_mode = mode
    new_shape = list(tensor.shape)

    if T.ndim(matrix_or_vector) == 2:  # Tensor times matrix
        # Test for the validity of the operation
        dim = 0 if transpose else 1
        if matrix_or_vector.shape[dim] != tensor.shape[mode]:
            raise ValueError(
                f"shapes {tensor.shape} and {matrix_or_vector.shape} not aligned in mode-{mode} multiplication: "
                f"{tensor.shape[mode]} (mode {mode}) != {matrix_or_vector.shape[dim]} (dim 1 of matrix)"
            )

        if transpose:
            matrix_or_vector = T.conj(T.transpose(matrix_or_vector))

        new_shape[mode] = matrix_or_vector.shape[0]
        vec = False

    elif T.ndim(matrix_or_vector) == 1:  # Tensor times vector
        if matrix_or_vector.shape[0] != tensor.shape[mode]:
            raise ValueError(
                f"shapes {tensor.shape} and {matrix_or_vector.shape} not aligned for mode-{mode} multiplication: "
                f"{tensor.shape[mode]} (mode {mode}) != {matrix_or_vector.shape[0]} (vector size)"
            )
        if len(new_shape) > 1:
            new_shape.pop(mode)
        else:
            new_shape = ()
        vec = True

    else:
        raise ValueError(
            "Can only take n_mode_product with a vector or a matrix."
            f"Provided array of dimension {T.ndim(matrix_or_vector)} not in [1, 2]."
        )

    res = T.dot(matrix_or_vector, unfold(tensor, mode))

    if vec:  # We contracted with a vector, leading to a vector
        return vec_to_tensor(res, shape=new_shape)
    else:  # tensor times vec: refold the unfolding
        return fold(res, fold_mode, new_shape)


def multi_mode_dot(tensor, matrix_or_vec_list, modes=None, skip=None, transpose=False):
    """n-mode product of a tensor and several matrices or vectors over several modes

    Parameters
    ----------
    tensor : ndarray

    matrix_or_vec_list : list of matrices or vectors of length ``tensor.ndim``

    skip : None or int, optional, default is None
        If not None, index of a matrix to skip.
        Note that in any case, `modes`, if provided, should have a length of ``tensor.ndim``

    modes : None or int list, optional, default is None

    transpose : bool, optional, default is False
        If True, the matrices or vectors in in the list are transposed.
        For complex tensors, the conjugate transpose is used.

    Returns
    -------
    ndarray
        tensor times each matrix or vector in the list at mode `mode`

    Notes
    -----
    If no modes are specified, just assumes there is one matrix or vector per mode and returns:

    :math:`\\text{tensor  }\\times_0 \\text{ matrix or vec list[0] }\\times_1 \\cdots \\times_n \\text{ matrix or vec list[n] }`

    See also
    --------
    mode_dot
    """
    if modes is None:
        modes = range(len(matrix_or_vec_list))

    decrement = 0  # If we multiply by a vector, we diminish the dimension of the tensor

    res = tensor

    # Order of mode dots doesn't matter for different modes
    # Sorting by mode shouldn't change order for equal modes
    factors_modes = sorted(zip(matrix_or_vec_list, modes), key=lambda x: x[1])
    for i, (matrix_or_vec, mode) in enumerate(factors_modes):
        if (skip is not None) and (i == skip):
            continue

        if transpose:
            res = mode_dot(res, T.conj(T.transpose(matrix_or_vec)), mode - decrement)
        else:
            res = mode_dot(res, matrix_or_vec, mode - decrement)

        if T.ndim(matrix_or_vec) == 1:
            decrement += 1

    return res